In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import sqlite3

from src.config import DATABASE_FILE

from src.quality.quality_checks import (
    load_quality_enabled_datasets,
    run_quality_checks,
)

from src.quality.quality_metrics import (
    calculate_quality_metrics,
)


# Phase 4 — Contrôle qualité des données

## Objectif

L'objectif de cette phase est de mettre en place un dispositif de contrôle
qualité permettant d'évaluer la fiabilité des données intégrées dans la
plateforme d'investissement.

La qualité des données constitue un élément fondamental dans les institutions
financières. Une donnée incorrecte peut conduire à :

- une analyse erronée ;
- une décision d'investissement inappropriée ;
- une perte financière ;
- un risque opérationnel ;
- un risque de conformité.

Cette phase introduit un modèle de contrôle qualité standardisé,
intégrable à tout nouveau dataset chargé dans la plateforme.

## Architecture du Framework

Le dispositif de contrôle qualité repose sur quatre composants :

1. dataset_registry
2. quality_checks.py
3. quality_metrics.py
4. data_control_center.py

Flux de traitement :

Source Registry
↓
Database Loader
↓
SQLite Database
↓
Dataset Registry
↓
Quality Checks
↓
Quality Metrics
↓
Data Control Center

## Jeux de données soumis au contrôle qualité

Le dispositif retient automatiquement les jeux de données dont :

- active = 1
- quality_enabled = 1

Les règles de qualité sont donc pilotées par le registre des sources,
et non par des noms de tables écrits dans le code.

In [ ]:
connection = sqlite3.connect(
    DATABASE_FILE
)

datasets = load_quality_enabled_datasets(
    connection
)

datasets

## Contrôles implémentés

Les contrôles suivent le brief de la phase 4. Chaque jeu dont `quality_enabled = 1` reçoit les six contrôles. Un contrôle sans objet sur la table reçoit le statut `NOT_APPLICABLE` et sort du score.

QC-001 Valeurs manquantes

Compte les lignes dont un champ obligatoire est vide. Sur le master, le symbole Yahoo et la devise de cotation sont obligatoires.

QC-002 Doublons

Détecte les doublons sur la clé métier : titre et place, symbole et date, série et date.

QC-003 Valeurs aberrantes

Repère un cours négatif ou hors de sa fourchette high-low, un taux de change nul ou négatif, et un SARON hors de l'intervalle retenu.

QC-004 Formats incorrects

Vérifie le format ISO des devises.

QC-005 Cohérence des devises

Vérifie que la devise de cotation correspond à la place. Un titre qui n'est pas déjà en CHF doit avoir une série de change cotée en CHF. La devise de benchmark du master peut différer de la devise de cotation.

QC-006 Chronologie

Lorsque le registre active la chronologie, les dates doivent se lire et ne pas être postérieures à la date du contrôle.


In [ ]:
from src.quality.quality_checks import (
    run_quality_checks,
)

controls = run_quality_checks()

controls

## KPI calculés

Le dispositif produit un taux par contrôle, puis un score par jeu de données :

- Valeurs manquantes
- Doublons
- Valeurs aberrantes
- Formats incorrects
- Cohérence des devises
- Chronologie
- Score du jeu
- Note
- Statut

Le score global est la moyenne non pondérée des jeux.


## Modèle de scoring

Chaque contrôle applicable produit un taux : la part des lignes sans anomalie, soit `(1 - anomalies / lignes) × 100`. Une ligne fautive ne met pas le contrôle à zéro.

- aucune anomalie : PASS
- au moins une anomalie et un taux d'au moins 80 : WARNING
- taux inférieur à 80, ou table vide : FAIL

Le seuil de 80 sépare un contrôle encore utilisable d'un contrôle cassé. Au-dessus, l'anomalie reste visible, elle ne fait pas échouer le jeu. C'est le cas des onze devises sans série de la Banque nationale et des quelques barres de cours écartées.

Le score d'un jeu est la moyenne de ses contrôles applicables. Un contrôle `NOT_APPLICABLE` n'entre pas : un contrôle sans objet ne doit pas ajouter un 100 artificiel. Le score global est la moyenne des jeux, chaque jeu ayant le même poids. Les 649 659 cours ne doivent pas cacher un référentiel de 120 lignes.

La note suit le score : A à partir de 97, B à partir de 90, C à partir de 80, sinon D. Le statut global est le moins bon statut de jeu. Une note A peut donc coexister avec un statut WARNING : le score est élevé, et des anomalies restent visibles dans le rapport.

Le registre ODD est enregistré dans le rapport et reste hors de cette moyenne. Un script peut vérifier que le fichier est complet et que le statut est un des quatre mots. Il ne peut pas décider si les phrases sont vraies.


In [ ]:
quality_results = calculate_quality_metrics(
    controls
)

quality_results["dataset_metrics"]


In [5]:
print(
    "Score:",
    quality_results[
        "global_quality_score"
    ]
)

print(
    "Grade:",
    quality_results[
        "global_quality_grade"
    ]
)

print(
    "Status:",
    quality_results[
        "global_quality_status"
    ]
)

Score: 99.40
Grade: A
Status: WARNING


## Champs non applicables

Un contrôle qui n'a pas d'objet sur une table n'entre pas dans le score. Exemples :

- la chronologie est désactivée pour le master ;
- les valeurs aberrantes ne s'appliquent pas au référentiel des titres ;
- la cohérence des devises s'applique au master et aux changes.

DS-006 reste hors du score. `quality_enabled = 0`. Au lancement de la phase 4, le contrôle du fichier vérifie une ligne par titre, l'absence de doublon sur le ticker et la place, le vocabulaire du statut, et les champs exigés d'une ligne Eligible. Il compte les numéros d'ODD imprimés, les controverses encore ouvertes et les parts de chiffre d'affaires énoncées. Le constat va dans la note du contrôle et dans le journal technique. Une part non imprimée n'est pas un manque. Ce constat n'entre pas dans le score.


## Livrables

Le dispositif produit le rapport, les indicateurs et le tableau de bord.

Le rapport qualité est la feuille `Quality Controls` de `audit/data_control_center.xlsx`, complétée par `Quality Issues` pour les anomalies. Les lignes d'autorité sont dans `quality_run`, `quality_control_result` et `quality_metric_result`. Le classeur en est l'export. Une modification faite à la main dans Excel ne change pas la base. La commande `python -m src.quality.data_control_center --export-only` réécrit le classeur depuis ces tables, sans relancer les contrôles.

Les KPI sont les six taux, le score, la note et le statut. Ils sont dans la feuille `Quality Dashboard` et dans `quality_metric_result`.

Le tableau de bord est le classeur `audit/data_control_center.xlsx` :

- Quality Dashboard
- Quality Controls
- Quality Issues
- Quality History

Les graphiques sont placés sous les tableaux. Le titre et les titres d'axes sont hors du tracé.

La commande du tableau de bord est :

```text
python -m src.quality.data_control_center
```

Les séries traitées sont un livrable distinct, publié par `python -m src.preparation.quality_treatment`. Elles sont décrites dans la section suivante. Leur vérification est dans `tests/quality/phase_04_validation.ipynb`.


## Traitement des anomalies

Le rapport qualité constate les écarts. Il ne réécrit pas les fichiers bruts. Un seul passage de préparation publie les séries que les phases suivantes partagent, afin que les trois portefeuilles, le suivi, la performance, le risque et les stress tests utilisent les mêmes rendements en CHF.

La commande est :

```text
python -m src.preparation.quality_treatment
```

Les règles sont les suivantes.

- Une barre de cours est écartée lorsque le close est absent, lorsqu'un prix ou le volume est négatif, ou lorsque l'open ou le close sort de la fourchette high-low au-delà de la tolérance du contrôle des valeurs aberrantes. Elle n'est pas remplacée par le close précédent. Le motif est écrit dans `data/02_processed/market/market_price_exclusions.csv`.
- Le close conservé est converti en CHF avec le taux SNB du mois du cours. Une cotation SNB publiée pour 100 unités est ramenée à une unité. Une place en CHF n'a pas besoin de taux. Un mois sans taux est écarté, sans interpolation.
- L'échelle du cours vient de la devise que Yahoo publie pour le symbole, dans `data/02_processed/market/quote_currency.csv`. Un code égal à la devise de cotation laisse le close inchangé. `GBp`, `ZAc` et `ILA` sont les unités mineures publiées par Yahoo pour la livre, le rand et le shekel : le close est alors divisé par 100. L'échelle reste dans `quote_scale`. Un autre code, ou une devise absente, écarte la barre au lieu de deviner l'échelle.
- Le taux du Trésor américain `DGS3MO` a été retiré du registre, de la base et des fichiers traités. Le seul taux sans risque est le SARON.
- Le taux sans risque en CHF est le SARON publié par la BNS, cube `zimoma`, série mensuelle depuis janvier 2009. Le fichier traité est `data/02_processed/macro/saron.csv`, avec `rate_currency` égal à CHF. C'est un taux au jour le jour, en pourcentage. C'est lui que le Sharpe utilisera.
- L'univers éligible en CHF est le master moins les cotations dont la devise n'a pas de série SNB. Le cube `devkum` ne publie pas `ILS`, `INR`, `SAR` ni `TWD`. Le fichier est `data/03_universe/eligible/chf_portfolio_universe.csv`. Aucun change croisé, aucune ligne obligataire et aucune ligne ETF n'est ajouté.

Les fichiers de `data/01_raw` ne sont pas modifiés. La vérification de ces fichiers est dans `tests/quality`, pas dans ce notebook.


In [6]:
from src.config import (
    ELIGIBLE_UNIVERSE,
    PROCESSED_MACRO,
    PROCESSED_MARKET,
)

prices = pd.read_csv(PROCESSED_MARKET / "market_prices_chf.csv")
exclusions = pd.read_csv(
    PROCESSED_MARKET / "market_price_exclusions.csv",
    dtype=str,
    keep_default_na=False,
)
saron = pd.read_csv(PROCESSED_MACRO / "saron.csv")
universe = pd.read_csv(
    ELIGIBLE_UNIVERSE / "chf_portfolio_universe.csv",
    dtype=str,
    keep_default_na=False,
)

print(f"Cours conservés : {len(prices):,}".replace(",", " "))
print("Barres écartées :")
print(exclusions["reason"].value_counts().to_string())
print()
print("Univers CHF :")
print(universe["chf_status"].value_counts().to_string())
print(
    "Devises sans série SNB :",
    sorted(universe.loc[universe["chf_status"].eq("excluded"), "listing_currency"].unique()),
)
print(f"SARON : {len(saron):,} observations".replace(",", " ") + f" ; devise : {saron['rate_currency'].unique().tolist()}")


Cours conservés : 587 816
Barres écartées :
reason
no_snb_series         50779
missing_monthly_fx     8202
outside_high_low       1739
missing_close          1123

Univers CHF :
chf_status
included    109
excluded     11
Devises sans série SNB : ['ILS', 'INR', 'SAR', 'TWD']
SARON : 213 observations ; devise : ['CHF']


## Conclusion

La phase 4 contrôle les jeux activés dans le registre, avec les six contrôles demandés, puis publie les séries traitées que les phases suivantes utiliseront.

Le passage du 5 novembre 2026 donne un score de 99,40, une note A et un statut WARNING. Les trois avertissements sont les onze titres sans série de la Banque nationale, les 1 123 séances sans cours et les 1 739 barres hors fourchette. Le registre ODD est contrôlé à part et ne compte pas dans ce score. Une anomalie reste visible dans le rapport même lorsque le taux du jeu reste élevé. Le statut global retient le moins bon jeu, afin qu'une note élevée ne masque pas les écarts.

Les fichiers bruts restent la source. Le traitement écarte les barres inutilisables et convertit les closes conservés en CHF, sans inventer un cours, un taux ou un identifiant.


## Suites possibles

- Règles d'aberration propres à d'autres concepts comptables
- Fraîcheur mesurée à partir de `last_acquisition_date` une fois cette date alimentée par le chargement
- Les cours de marché contrôlés sont ceux de Yahoo, DS-003.


# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas.

**BNS (SNB).** Banque nationale suisse. Source du taux de change du mois et du SARON.

**Close.** Cours de clôture. Le close conservé est converti en francs.

**Cohérence des devises (QC-005).** Contrôle : la devise de cotation correspond à la place, et un titre qui n'est pas en CHF a une série de change en CHF.

**Chronologie (QC-006).** Contrôle : les dates se lisent et ne sont pas postérieures à la date du contrôle.

**Doublons (QC-002).** Lignes répétées sur la clé métier.

**DS-006.** Jeu de l'évaluation ODD. `quality_enabled = 0`. Le contrôle de fichier est consigné hors score. Il n'exige pas une part de chiffre d'affaires.

**FAIL.** Statut d'un contrôle dont le taux est inférieur à 80, ou d'une table vide.

**Formats incorrects (QC-004).** Contrôle du format ISO des devises.

**GBp, ZAc, ILA.** Unités mineures publiées par Yahoo pour la livre, le rand et le shekel. Le close est alors divisé par 100.

**Interpolation.** Estimation d'un taux absent. Cette note ne la fait pas : un mois sans taux est écarté.

**NOT_APPLICABLE.** Statut d'un contrôle sans objet sur la table. Il sort du score.

**Note.** Lettre dérivée du score : A à partir de 97, B à partir de 90, C à partir de 80, sinon D.

**PASS.** Statut d'un contrôle sans anomalie.

**SARON.** Taux au jour le jour du franc publié par la BNS. Seul taux sans risque retenu. Le taux américain `DGS3MO` a été retiré.

**Score.** Moyenne des taux des contrôles applicables. Le score global est la moyenne des jeux, sans pondérer par le nombre de lignes.

**Statut.** PASS, WARNING ou FAIL. Le statut global est le moins bon statut de jeu.

**Valeurs aberrantes (QC-003).** Cours négatif ou hors de la fourchette high-low, change nul ou négatif, indicateur macro hors de son intervalle.

**Valeurs manquantes (QC-001).** Champ obligatoire vide.

**WARNING.** Au moins une anomalie, avec un taux d'au moins 80.

**Yahoo.** Source des cours contrôlés, jeu `DS-003`.
